In [1]:
import torch
import torch_pruning as tp

from torch.nn.utils import prune
from torchvision.models import resnet18
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

# Модель для измерения качества

In [2]:
seed = 12345
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_epochs = 5
generator = torch.manual_seed(seed)


dataset = CIFAR10(
    root='data/',
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        )
    ])
)
dataset_test = CIFAR10(
    root='data/',
    train=False,
    download=False,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        )
    ])
)
dataset_train, dataset_val = torch.utils.data.random_split(
    dataset,
    [0.8, 0.2],
    generator
)

dataloader_train = DataLoader(
    dataset_train,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    generator=generator,
)
dataloader_val = DataLoader(
    dataset_val,
    batch_size=64,
    num_workers=4,
    shuffle=False
)
dataloader_test = DataLoader(dataset_test, batch_size=64, shuffle=False)

samples_train = len(dataset_train)
samples_val = len(dataset_val)

model = resnet18(weights=None)
model.fc = torch.nn.Linear(512, 10)
model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=1)
model.maxpool = torch.nn.Identity()
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kern

In [3]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.002)


def train():
    for epoch in range(n_epochs):
        loss_train = 0
        loss_val = 0
        samples_correct = 0

        model.train()
        for i_batch_train, batch in enumerate(tqdm(dataloader_train)):
            images = batch[0].to(device)
            labels = batch[1].to(device)

            optimizer.zero_grad()

            output = model(images)

            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

            loss_train += loss.item() * images.size(0)

        model.eval()
        with torch.no_grad():
            for i_batch_val, batch in enumerate(tqdm(dataloader_val)):
                images = batch[0].to(device)
                labels = batch[1].to(device)

                output = model(images)
                loss = criterion(output, labels)

                loss_val += loss.item() * images.size(0)
                samples_correct += (labels == output.argmax(dim=1)).sum().item()

        print(
            f' epoch: {epoch}\t'
            f'train loss: {loss_train / samples_train}\t'
            f'val loss: {loss_val / samples_val}\t'
            f'val accuracy: {samples_correct / samples_val} ({samples_correct}/{samples_val})'
        )

    torch.save(model.state_dict(), 'data/model.pth')


if __name__ == '__main__':
    if not os.path.exists('data/model.pth'):
        train()

In [4]:
model.load_state_dict(
    torch.load(
        'data/model.pth',
        weights_only=True,
        map_location=device
    )
)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kern

## Магнитудный неструктурированный прунинг

In [5]:
class MagnitudePruner(prune.BasePruningMethod):
    PRUNING_TYPE = 'unstructured'

    def __init__(self, percentage, *params):
        super().__init__(*params)
        self.percentage = percentage

    def compute_mask(self, t, default_mask):
        weights = t.flatten().abs()
        weights, _ = torch.sort(weights)
        n_params = weights.shape[0]
        threshold = weights[int(n_params * self.percentage)]
        mask = t.abs() > threshold
        return mask

In [6]:
model = torch.nn.Linear(5, 5)
print('before pruning:\n', list(model.named_parameters()))

MagnitudePruner.apply(model, name='weight', percentage=0.5)

print('after pruning:\n', model.weight)

before pruning:
 [('weight', Parameter containing:
tensor([[ 0.1928, -0.1141, -0.0519, -0.4236, -0.4020],
        [-0.3402,  0.4038, -0.0582,  0.0942, -0.4255],
        [-0.2277,  0.4365,  0.1880,  0.3892, -0.4041],
        [ 0.4091, -0.4058, -0.2517,  0.3798, -0.0767],
        [ 0.4230,  0.0810, -0.1507,  0.0206,  0.1713]], requires_grad=True)), ('bias', Parameter containing:
tensor([ 0.3680,  0.3534, -0.0757,  0.1469, -0.2792], requires_grad=True))]
after pruning:
 tensor([[ 0.0000, -0.0000, -0.0000, -0.4236, -0.4020],
        [-0.3402,  0.4038, -0.0000,  0.0000, -0.4255],
        [-0.0000,  0.4365,  0.0000,  0.3892, -0.4041],
        [ 0.4091, -0.4058, -0.0000,  0.3798, -0.0000],
        [ 0.4230,  0.0000, -0.0000,  0.0000,  0.0000]], grad_fn=<MulBackward0>)


## Магнитудный структурированный прунинг

In [7]:
class ExampleModel(torch.nn.Module):
    def __init__(self, in_size=3, hidden_size=5, out_size=3):
        super().__init__()
        self.hidden_size = hidden_size

        self.conv = torch.nn.Conv2d(in_size, hidden_size, kernel_size=3)
        self.pooling = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.relu = torch.nn.ReLU()
        self.fc1 = torch.nn.Linear(hidden_size, hidden_size)
        self.fc2 = torch.nn.Linear(hidden_size, out_size)

    def forward(self, x):
        x = self.pooling(self.relu(self.conv(x))).reshape(-1, self.hidden_size)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return x

In [8]:
class MagnitudeStructuredPruner:

    @staticmethod
    def apply(model, module, pruning_fn, percentage, example_inputs):
        dg = tp.DependencyGraph().build_dependency(
            model,
            example_inputs=example_inputs
        )

        t = module.weight.abs()
        t = t.reshape(t.shape[0], -1)
        t = t.mean(dim=1)
        t, _ = torch.sort(t)
        n_params = t.shape[0]
        threshold = t[int(n_params * percentage)].item()
        idxs = (t < threshold).nonzero(as_tuple=True)[0]

        group = dg.get_pruning_group(module, pruning_fn, idxs=idxs)

        if dg.check_pruning_group(group):
            group.prune()

In [9]:
for images, labels in dataloader_train:
    print(images.shape, labels.shape)
    break

torch.Size([64, 3, 32, 32]) torch.Size([64])


In [10]:
model = ExampleModel()
print('before pruning:', model)
MagnitudeStructuredPruner.apply(
    model,
    model.fc1,
    tp.prune_linear_out_channels,
    percentage=0.5,
    example_inputs=torch.randn(3, 3, 32, 32)
)
MagnitudeStructuredPruner.apply(
    model,
    model.conv,
    tp.prune_conv_out_channels,
    percentage=0.5,
    example_inputs=torch.randn(3, 3, 32, 32)
)

print('after pruning:', model)

before pruning: ExampleModel(
  (conv): Conv2d(3, 5, kernel_size=(3, 3), stride=(1, 1))
  (pooling): AdaptiveAvgPool2d(output_size=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=5, out_features=5, bias=True)
  (fc2): Linear(in_features=5, out_features=3, bias=True)
)
after pruning: ExampleModel(
  (conv): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1))
  (pooling): AdaptiveAvgPool2d(output_size=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=3, out_features=3, bias=True)
  (fc2): Linear(in_features=3, out_features=3, bias=True)
)


## Градиентный неструктурированный прунинг

In [11]:
n_classes = 5
n_images = 20

model = ExampleModel(hidden_size=4, out_size=n_classes)

example_inputs = torch.randn(n_images, 3, 32, 32)
example_labels = torch.randint(low=0, high=3, size=(n_images,))
batch_size = 3

criterion = torch.nn.CrossEntropyLoss(reduction='sum')

for i in range(int(example_inputs.size(0) / 3) + 1):
    images = example_inputs[
        batch_size * i:
        min(batch_size * (i + 1), example_inputs.size(0))
    ]
    labels = example_labels[
        batch_size * i:
        min(batch_size * (i + 1), example_inputs.size(0))
    ]
    output = model(images)

    loss = criterion(output, labels)
    loss.backward()

grad = model.fc1.weight.grad / n_images

print('fc1 mean gradient:')
print(grad)

fc1 mean gradient:
tensor([[ 0.0095,  0.0173,  0.0155,  0.0179],
        [ 0.0146,  0.0265,  0.0238,  0.0274],
        [-0.0071, -0.0129, -0.0116, -0.0134],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])


In [12]:
class GradientPruner(prune.BasePruningMethod):
    PRUNING_TYPE = 'unstructured'

    def __init__(self, percentage, t_gradient, *params):
        super().__init__(*params)
        self.percentage = percentage
        self.t_gradient = t_gradient

    def compute_mask(self, t, default_mask):
        abs_grad_list = self.t_gradient.flatten().abs()
        abs_grad_list, _ = torch.sort(abs_grad_list)
        n_params = abs_grad_list.shape[0]
        threshold = abs_grad_list[int(n_params * self.percentage)]
        mask = self.t_gradient.abs() > threshold
        return mask

In [13]:

GradientPruner.apply(model.fc1, name='weight', percentage=0.5, t_gradient=grad)

print('after pruning:\n', model.fc1.weight)

abs_grad_list = grad.flatten().abs()
abs_grad_list, _ = torch.sort(abs_grad_list)
n_params = abs_grad_list.shape[0]
threshold = abs_grad_list[int(n_params * 0.5)]
mask = grad.abs() > threshold

print('mask:\n', mask * 1)
print('gradient:\n', grad)
print('threshold:', threshold)

after pruning:
 tensor([[ 0.0000, -0.0800,  0.4205, -0.0627],
        [-0.1938,  0.1006, -0.4395, -0.3608],
        [-0.0000,  0.0000,  0.0000, -0.0000],
        [-0.0000, -0.0000,  0.0000,  0.0000]], grad_fn=<MulBackward0>)
mask:
 tensor([[0, 1, 1, 1],
        [1, 1, 1, 1],
        [0, 0, 0, 0],
        [0, 0, 0, 0]])
gradient:
 tensor([[ 0.0095,  0.0173,  0.0155,  0.0179],
        [ 0.0146,  0.0265,  0.0238,  0.0274],
        [-0.0071, -0.0129, -0.0116, -0.0134],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])
threshold: tensor(0.0134)
